In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

# ค้นหาไฟล์ข้อมูลจากตำแหน่งที่ใช้บ่อยในโปรเจกต์
file_name = "dataset - 2020-09-24.csv"
path_candidates = [
    Path.cwd() / file_name,
    Path.cwd().parent / file_name,
    Path.cwd().parent / "dataset" / file_name,
]
DATA_PATH = next((path for path in path_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"ไม่พบไฟล์ {file_name} ในตำแหน่ง {path_candidates}")
dataset = pd.read_csv(DATA_PATH)

# ลบเครื่องหมายเปอร์เซ็นต์และแปลงค่าตัวเลข
percent_columns = [
    "Tackle success %", "Shooting accuracy %", "Cross accuracy %"
]
for column in percent_columns:
    dataset[column] = pd.to_numeric(
        dataset[column].astype("string").str.rstrip("%"), errors="coerce"
    ) / 100

categorical_columns = ["Club", "Position", "Nationality"]
for column in categorical_columns:
    dataset[column] = dataset[column].astype("string").fillna("Unknown")

for column in dataset.columns:
    if column not in categorical_columns + ["Name"] + percent_columns:
        dataset[column] = pd.to_numeric(dataset[column], errors="coerce")

# ทำนายจำนวนประตู โดยตัดคอลัมน์ที่ทำให้เกิด target leakage
TARGET_COLUMN = "Goals"
leakage_columns = [
    "Goals", "Goals per match", "Headed goals", "Goals with right foot",
    "Goals with left foot", "Own goals"
]
feature_columns = [
    column for column in dataset.columns
    if column not in leakage_columns + ["Name"]
]
model_data = dataset[feature_columns + [TARGET_COLUMN]].dropna(subset=[TARGET_COLUMN])
X = model_data[feature_columns]
y = model_data[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [
    column for column in X.columns if column not in numeric_features
]

# SVM ต้องใช้ข้อมูลตัวเลขที่ผ่านการปรับสเกล
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

model = SVR(
    kernel="rbf",
    C=10.0,
    epsilon=0.1,
    gamma="scale",
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])
pipeline.fit(X_train, y_train)

predictions = np.maximum(pipeline.predict(X_test), 0)
metrics = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
    "R2": r2_score(y_test, predictions),
}

print(f"โมเดล: {model.__class__.__name__}")
print(f"ไฟล์ข้อมูล: {DATA_PATH}")
print(f"จำนวนข้อมูลทั้งหมด: {len(model_data):,} แถว")
print(f"ข้อมูลฝึกสอน: {len(X_train):,} แถว | ข้อมูลทดสอบ: {len(X_test):,} แถว")
print("ผลการประเมินโมเดล:")
print(f"  MAE (ค่าคลาดเคลื่อนสัมบูรณ์เฉลี่ย): {metrics['MAE']:.4f}")
print(f"  RMSE (รากที่สองของค่าคลาดเคลื่อนกำลังสองเฉลี่ย): {metrics['RMSE']:.4f}")
print(f"  R² (ความสามารถในการอธิบายข้อมูล): {metrics['R2']:.4f}")

# คำนวณความสำคัญของตัวแปรด้วย permutation importance แบบไม่ใช้ sklearn.inspection
base_mae = mean_absolute_error(y_test, predictions)
rng = np.random.default_rng(42)
permutation_scores = {}
for column in X_test.columns:
    shuffled_test = X_test.copy()
    shuffled_test[column] = rng.permutation(shuffled_test[column].to_numpy())
    shuffled_predictions = np.maximum(pipeline.predict(shuffled_test), 0)
    permutation_scores[column] = (
        mean_absolute_error(y_test, shuffled_predictions) - base_mae
    )
importance = pd.Series(permutation_scores).sort_values(ascending=False)
print("\nตัวแปรสำคัญ 15 อันดับแรก:")
print(importance.head(15).to_string())

results = X_test[["Club", "Position"]].copy()
results["ประตูจริง"] = y_test
results["ประตูที่โมเดลทำนาย"] = predictions.round(2)
print("\nตัวอย่างผลการทำนาย:")
print(results.head(10).to_string())

โมเดล: SVR
ไฟล์ข้อมูล: c:\Users\guyza\OneDrive\Desktop\PML\PML\dataset\dataset - 2020-09-24.csv
จำนวนข้อมูลทั้งหมด: 571 แถว
ข้อมูลฝึกสอน: 456 แถว | ข้อมูลทดสอบ: 115 แถว
ผลการประเมินโมเดล:
  MAE (ค่าคลาดเคลื่อนสัมบูรณ์เฉลี่ย): 1.8840
  RMSE (รากที่สองของค่าคลาดเคลื่อนกำลังสองเฉลี่ย): 4.5748
  R² (ความสามารถในการอธิบายข้อมูล): 0.8595

ตัวแปรสำคัญ 15 อันดับแรก:
Blocked shots          0.476478
Wins                   0.450336
Shots on target        0.435234
Offsides               0.433573
Shots                  0.431587
Hit woodwork           0.336738
Aerial battles won     0.266620
Fouls                  0.251276
Big chances missed     0.247710
Yellow cards           0.243971
Assists                0.240768
Successful 50/50s      0.206637
Appearances            0.197648
Freekicks scored       0.191449
Aerial battles lost    0.171702

ตัวอย่างผลการทำนาย:
                         Club    Position  ประตูจริง  ประตูที่โมเดลทำนาย
509      West-Bromwich-Albion  Midfielder          0             